# Experiment 1: Adaptive Linear Neuron (ADALINE)

## Overview
ADALINE (Adaptive Linear Neuron) is a single-layer neural network that uses a linear activation function and learns using the Widrow-Hoff learning rule (delta rule). Unlike the perceptron, ADALINE uses the actual linear output for error calculation instead of the class label.

## Key Concepts
- **Linear Activation Function**: Output = weighted sum of inputs
- **Delta Rule**: Updates weights based on the difference between target and linear output
- **Cost Function**: Sum of Squared Errors (SSE)
- **Learning Rate**: Controls the magnitude of weight updates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## ADALINE Implementation

In [ ]:
class ADALINEClassifier:
    """Adaptive Linear Neuron classifier"""
    
    def __init__(self, learning_rate=0.01, n_iter=50, random_state=1):
        """
        Initialize ADALINE classifier
        
        Parameters:
        -----------
        learning_rate : float, default=0.01
            Learning rate for weight updates (eta)
        n_iter : int, default=50
            Number of passes over the training dataset
        random_state : int, default=1
            Random seed for weight initialization
        """
        self.eta = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state
        self.cost_ = []
        self.weights_ = None
        self.bias_ = None
    
    def fit(self, X, y):
        """
        Fit the ADALINE classifier to training data
        
        Parameters:
        -----------
        X : array-like, shape = [n_samples, n_features]
            Training vectors
        y : array-like, shape = [n_samples]
            Target values
        
        Returns:
        --------
        self : object
        """
        rgen = np.random.RandomState(self.random_state)
        self.weights_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.bias_ = 0.0
        
        for _ in range(self.n_iter):
            # Compute linear output
            output = self.net_input(X)
            
            # Compute error
            errors = (y - output)
            
            # Update weights using delta rule
            self.weights_ += self.eta * X.T.dot(errors)
            self.bias_ += self.eta * errors.sum()
            
            # Calculate cost (Sum of Squared Errors)
            cost = (errors ** 2).sum() / 2.0
            self.cost_.append(cost)
        
        return self
    
    def net_input(self, X):
        """Calculate linear output (activation)"""
        return np.dot(X, self.weights_) + self.bias_
    
    def predict(self, X):
        """Return class label after unit step"""
        return np.where(self.net_input(X) >= 0.5, 1, 0)
    
    def score(self, X, y):
        """Calculate accuracy score"""
        return accuracy_score(y, self.predict(X))

## Dataset and Training

In [ ]:
# Generate sample data
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, 
                            n_redundant=0, random_state=42)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"Class distribution in training set: {np.bincount(y_train)}")

## Train ADALINE

In [ ]:
# Train ADALINE classifier
adaline = ADALINEClassifier(learning_rate=0.01, n_iter=50, random_state=1)
adaline.fit(X_train_scaled, y_train)

# Get predictions
y_train_pred = adaline.predict(X_train_scaled)
y_test_pred = adaline.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\n=== ADALINE Performance ===")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=['Class 0', 'Class 1']))

## Visualization

In [ ]:
# Plot cost function over iterations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cost function
ax1.plot(range(1, len(adaline.cost_) + 1), adaline.cost_, marker='o', linestyle='-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Sum of Squared Errors (SSE)')
ax1.set_title('ADALINE - Cost Function During Training')
ax1.grid(True, alpha=0.3)

# Plot 2: Decision boundary
h = 0.02
x_min, x_max = X_test_scaled[:, 0].min() - 1, X_test_scaled[:, 0].max() + 1
y_min, y_max = X_test_scaled[:, 1].min() - 1, X_test_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = adaline.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
ax2.scatter(X_test_scaled[y_test == 0, 0], X_test_scaled[y_test == 0, 1], 
            label='Class 0', marker='o', color='red', edgecolors='k')
ax2.scatter(X_test_scaled[y_test == 1, 0], X_test_scaled[y_test == 1, 1], 
            label='Class 1', marker='s', color='blue', edgecolors='k')
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('ADALINE - Decision Boundary')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nDecision boundary plot shows the classifier's separation of classes.")

## Summary

**ADALINE Key Points:**
- Uses linear activation function instead of step function
- Applies delta rule (Widrow-Hoff) for weight updates
- Cost function is Sum of Squared Errors (SSE)
- Converges to optimal solution when data is linearly separable
- Learning rate and number of iterations are important hyperparameters
- Feature scaling improves convergence

## Viva Questions

### Basic Concepts
1. **What is ADALINE and how does it differ from the Perceptron?**
   - ADALINE uses linear activation function and delta rule, while Perceptron uses step function
   - ADALINE calculates error on linear output, not on class label

2. **Explain the Widrow-Hoff learning rule (Delta Rule).**
   - Weights are updated based on difference between target and linear output
   - Formula: w = w + η(y - ŷ)x where ŷ is linear output

3. **What is the cost function used in ADALINE?**
   - Sum of Squared Errors (SSE): J = 1/2 Σ(y - ŷ)²

4. **What is the role of learning rate (eta)?**
   - Controls step size of weight updates
   - Too high: may overshoot optimal solution
   - Too low: convergence is very slow

### Implementation Questions
5. **Why is feature scaling important in ADALINE?**
   - Ensures faster convergence
   - Prevents certain features from dominating due to scale differences
   - StandardScaler is commonly used

6. **How do we decide the number of iterations?**
   - Based on convergence of cost function
   - Can use early stopping if cost stops decreasing
   - Usually determined experimentally

7. **Can ADALINE handle non-linearly separable data?**
   - No, it converges only for linearly separable data
   - For non-linear problems, use multi-layer neural networks

### Comparison Questions
8. **Compare ADALINE with Logistic Regression.**
   - ADALINE uses linear output with delta rule
   - Logistic Regression uses sigmoid activation with cross-entropy loss
   - Both are linear classifiers

9. **When should you use ADALINE over other algorithms?**
   - For linearly separable binary classification problems
   - When interpretability is important (linear model)
   - Educational purpose to understand neural networks

10. **What are limitations of ADALINE?**
    - Only works for linearly separable data
    - Binary classification only
    - Sensitive to feature scaling
    - Requires careful tuning of learning rate